<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 K-Means: Abrir Tiendas en un Pueblo 🫓
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 04
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/04%20-%20Clustering%20No%20Supervisado/Para%20Dummies/01_KMeans_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del [cuaderno 01 (K-Means)](../01_KMeans.ipynb). *K-Means* es el método más popular para **formar grupos**: parece magia, pero en el fondo es un truco muy sencillo que se repite hasta que nada cambia. Lo vamos a ver con una historia cotidiana: **abrir tiendas en un pueblo**.

Al terminar podrás explicar:
1. Los **dos pasos** que repite *K-Means* (asignar y mover) y por qué siempre termina.
2. Por qué el resultado puede depender de **la suerte del inicio** y cómo se arregla.
3. Cómo elegir **cuántos grupos** formar con el «método del codo».
4. **Cuándo se equivoca** *K-Means*: formas raras, grupos de distinto tamaño y clientes muy lejanos.

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../01_KMeans.ipynb). Otras miradas sencillas: [K-Means (Dummies) en Data Mining](../../../Data%20Mining/03%20-%20Clustering%20y%20Mineria%20Reglas%20de%20Asociacion/Para%20Dummies/00_KMeans_Dummies.ipynb) y [K-Means (Dummies) en Data Science Programming](../../../Data%20Science%20programming/10%20-%20Clustering/Para%20Dummies/02_KMeans_Clustering_Dummies.ipynb).

---
## 1. La historia de las tiendas de arepas 🫓

Un pueblo tiene **200 casas** y quieres abrir **3 tiendas** de arepas para que la gente camine lo menos posible. ¿Dónde las pones? *K-Means* lo resuelve así:

1. **Pon las 3 tiendas en lugares al azar** (por ejemplo, en 3 casas cualquiera).
2. **Cada casa va a la tienda que le queda más cerca.** (Así se forman 3 grupos de clientes.)
3. **Cada tienda se muda al centro de sus clientes** (al promedio de sus posiciones).
4. **Repite los pasos 2 y 3** hasta que **nadie cambie de tienda**.

Cada vez que haces 2 y 3, la distancia total que camina la gente **solo puede bajar o quedarse igual**. Por eso el método **siempre termina**. Miremos cómo se mueven las tiendas, ronda por ronda.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

casas, _ = make_blobs(n_samples=200, centers=3, cluster_std=1.1, random_state=3)       # 200 casas de un pueblo
rng = np.random.RandomState(1)
tiendas = casas[rng.choice(len(casas), 3, replace=False)].copy()                       # 1) 3 tiendas en casas al azar

fotos = []                                                                             # guardamos "fotos" de cada ronda
for ronda in range(1, 15):
    # 2) cada casa va a la tienda MAS CERCANA
    distancias = np.linalg.norm(casas[:, None, :] - tiendas[None, :, :], axis=2)
    cliente_de = distancias.argmin(axis=1)
    fotos.append((tiendas.copy(), cliente_de.copy()))
    # 3) cada tienda se mueve al CENTRO (promedio) de sus clientes
    nuevas = np.array([casas[cliente_de == k].mean(axis=0) for k in range(3)])
    if np.allclose(nuevas, tiendas):                                                   # 4) si nadie se mueve, terminamos
        print(f"¡Listo! Las tiendas dejaron de moverse en la ronda {ronda}.")
        break
    tiendas = nuevas

cuales = [0, 1, 2, len(fotos) - 1]
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, i in zip(axes, cuales):
    pos, clientes = fotos[i]
    ax.scatter(casas[:, 0], casas[:, 1], c=clientes, cmap="tab10", s=14, alpha=0.7)
    ax.scatter(pos[:, 0], pos[:, 1], marker="*", s=350, color="gold", edgecolor="black", linewidth=1.2)
    ax.set_title("Ronda " + str(i + 1) + (" (final)" if i == len(fotos) - 1 else ""), fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Las estrellas son las tiendas: se van moviendo al centro de sus clientes", fontweight="bold")
plt.tight_layout(); plt.show()

---
## 2. La suerte del inicio 🎲

El método **siempre termina**, pero no siempre termina en **el mejor lugar**. Si por mala suerte pones **dos tiendas en el mismo barrio** y **ninguna en otro**, el método se queda atascado ahí: cada tienda ya está «en el centro de sus clientes», aunque haya una solución mucho mejor.

Lo comprobamos con un pueblo de **8 barrios**: repetimos el método 12 veces, cada vez con un inicio al azar distinto, y anotamos **cuántos kilómetros en total caminaría la gente** (el programa lo llama *inercia*; menos es mejor).

Hay dos remedios sencillos:
- **Repetir varias veces y quedarse con la mejor** (en `scikit-learn`, el parámetro `n_init`).
- **Un inicio más inteligente (*k-means++*)**: poner cada tienda nueva **lejos de las que ya pusiste**, para no repetir barrio.

In [ ]:
from sklearn.cluster import KMeans

pueblo, _ = make_blobs(n_samples=600, centers=8, cluster_std=0.9, random_state=11)       # un pueblo con 8 barrios

al_azar = [KMeans(8, init="random", n_init=1, random_state=s).fit(pueblo).inertia_ for s in range(12)]
inteligente = [KMeans(8, init="k-means++", n_init=1, random_state=s).fit(pueblo).inertia_ for s in range(12)]

print("Distancia total de la gente (menos es mejor), en 12 intentos:")
print("  Tiendas puestas al azar     :", [int(v) for v in al_azar])
print("  Inicio inteligente (k-means++):", [int(v) for v in inteligente])
print(f"\nMejor y peor con inicio al azar : {min(al_azar):.0f} y {max(al_azar):.0f}")
print(f"Mejor y peor con inicio inteligente: {min(inteligente):.0f} y {max(inteligente):.0f}")
print("Con mala suerte, el resultado puede ser mucho peor. Por eso se repite varias veces y se usa el inicio inteligente (es lo que hace KMeans por defecto).")

---
## 3. ¿Cuántas tiendas abro? El método del codo 💪

Si pones **más tiendas**, la gente camina menos (con una tienda por casa, no camina nada). Pero cada tienda cuesta dinero. La pregunta es: **¿a partir de cuántas tiendas ya no vale la pena?**

Se dibuja la distancia total contra el número de tiendas. La curva baja mucho al principio y luego **se aplana**. El punto donde cambia (el «codo» del brazo) suele ser una buena elección: **agregar una tienda más ya casi no ayuda**.

In [ ]:
barrios4, _ = make_blobs(n_samples=500, centers=[[-6, -4], [6, -4], [-6, 5], [6, 5]], cluster_std=1.3, random_state=0)   # un pueblo con 4 barrios claros
tiendas_probadas = list(range(1, 10))
distancia_total = [KMeans(k, n_init=10, random_state=0).fit(barrios4).inertia_ for k in tiendas_probadas]

plt.figure(figsize=(8, 4.5))
plt.plot(tiendas_probadas, distancia_total, "o-", color="#2563eb", lw=2)
plt.axvline(4, color="#dc2626", ls="--", label="el 'codo': 4 tiendas")
plt.xlabel("Número de tiendas (grupos)"); plt.ylabel("Distancia total que camina la gente")
plt.title("Después de 4 tiendas, cada tienda extra ayuda muy poco", fontweight="bold"); plt.legend(); plt.show()
print("Mejora al pasar de 3 a 4 tiendas :", int(distancia_total[2] - distancia_total[3]))
print("Mejora al pasar de 4 a 5 tiendas :", int(distancia_total[3] - distancia_total[4]))
print("\nA veces el codo no es tan claro (si los barrios se mezclan). Entonces se usan otras notas, que veremos en el último cuaderno.")

---
## 4. Cuando K-Means se equivoca ⚠️

*K-Means* imagina cada grupo como una **zona redonda alrededor de su tienda**: la gente va a la tienda más cercana, así que las fronteras son **líneas rectas**. Eso funciona muy bien con barrios «redondos», pero falla en tres situaciones:

| Situación | Por qué falla | Ejemplo cotidiano |
|---|---|---|
| **Formas raras** (lunas, anillos) | Solo sabe hacer zonas redondas, cortadas con rectas | Un río con forma de media luna |
| **Grupos de distinto tamaño** | Tiende a «repartir» por igual: parte el grupo grande | Una ciudad grande junto a dos pueblitos |
| **Clientes muy lejanos** | La tienda se muda al **promedio**, y un cliente remoto la **arrastra** | Una finca lejanísima que desplaza la tienda |

Veámoslo. En cada gráfico medimos el **acuerdo** con los grupos reales (1 = perfecto; 0 = como adivinar al azar).

In [ ]:
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score

lunas, y_lunas = make_moons(n_samples=300, noise=0.07, random_state=0)
desiguales, y_des = make_blobs(n_samples=[500, 80, 80], centers=[[-5, 0], [1, 0], [5, 3]], cluster_std=[2.4, 0.9, 0.5], random_state=42)
base, y_base = make_blobs(n_samples=300, centers=[[-5, -3], [5, -3], [0, 5]], cluster_std=1.1, random_state=42)
con_lejanos = np.vstack([base, np.array([35, 35]) + np.array([[0, 0], [1, -1], [-1, 1], [0.5, 0.5], [-0.5, -0.5]])])
y_lej = np.r_[y_base, [3] * 5]                                                    # (los 5 lejanos llevan etiqueta propia)

casos = [("Formas raras (lunas)", lunas, y_lunas, 2), ("Grupos de distinto tamaño", desiguales, y_des, 3), ("5 clientes muy lejanos", con_lejanos, y_lej, 3)]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, (titulo, X, y_real, k) in zip(axes, casos):
    etiquetas = KMeans(k, n_init=10, random_state=0).fit_predict(X)
    ax.scatter(X[:, 0], X[:, 1], c=etiquetas, cmap="tab10", s=12, alpha=0.8)
    n_ref = len(base) if "lejanos" in titulo else len(X)                      # en el 3.o medimos solo con los clientes normales
    acuerdo = adjusted_rand_score(y_real[:n_ref], etiquetas[:n_ref])
    ax.set_title(f"{titulo}\nacuerdo con lo real = {acuerdo:.2f}", fontweight="bold", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("En los tres casos K-Means da un resultado engañoso: corta las lunas con una recta, parte el grupo grande y gasta una tienda en los clientes lejanos.")
print("(En el tercero medimos el acuerdo solo con los clientes normales.) Para estos casos existen otros métodos: el jerárquico y DBSCAN, en los siguientes cuadernos.")

---
##### 🛠️ Práctica 1: Abre las 4 tiendas del pueblo de los barrios claros

Volvamos al pueblo de **4 barrios claros** (`barrios4`) del método del codo. Ya sabemos que el codo estaba en 4 tiendas; ahora vamos a **abrirlas** de verdad:

1. Pídele a `KMeans` que forme **4 grupos** (`n_clusters=4`, `n_init=10`, `random_state=0`) con los datos `barrios4` y guárdalo en `modelo`.
2. Las posiciones de las tiendas están en `modelo.cluster_centers_`: guárdalas en `tiendas_4`. Cuenta cuántas casas atiende cada tienda (`np.bincount(modelo.labels_)`) y guárdalo en `clientes_por_tienda`.
3. Mira la **distancia total** que camina la gente (`modelo.inertia_`) y guárdala en `caminata_4`.
4. Comprueba que hay 4 tiendas, que cada una atiende entre 100 y 150 casas, y que `caminata_4` es igual al valor que ya calculamos para 4 tiendas en `distancia_total`.

In [ ]:
# Escribe tu código aquí

# 1) Abrir las 4 tiendas
# modelo = KMeans(n_clusters=..., n_init=10, random_state=0).fit(barrios4)

# 2) Donde quedaron y a cuantas casas atiende cada una
# tiendas_4 = modelo.cluster_centers_
# clientes_por_tienda = np.bincount(modelo.labels_)

# 3) Distancia total que camina la gente
# caminata_4 = modelo.inertia_

# 4) Comprobaciones (el 4.o valor de distancia_total corresponde a 4 tiendas)
# assert len(tiendas_4) == 4
# assert ((clientes_por_tienda >= 100) & (clientes_por_tienda <= 150)).all()
# assert np.isclose(caminata_4, distancia_total[3])


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# 1) Abrir las 4 tiendas
modelo = KMeans(n_clusters=4, n_init=10, random_state=0).fit(barrios4)

# 2) Donde quedaron y a cuantas casas atiende cada una
tiendas_4 = modelo.cluster_centers_
clientes_por_tienda = np.bincount(modelo.labels_)
print("Posiciones de las tiendas:\n", tiendas_4.round(1))
print("Casas por tienda:", clientes_por_tienda.tolist())

# 3) Distancia total que camina la gente
caminata_4 = modelo.inertia_
print(f"Distancia total con 4 tiendas: {caminata_4:.0f}")

# 4) Comprobaciones (distancia_total[3] es la del codo, con 4 tiendas)
assert len(tiendas_4) == 4
assert ((clientes_por_tienda >= 100) & (clientes_por_tienda <= 150)).all()
assert np.isclose(caminata_4, distancia_total[3])
print("Cuatro tiendas, una por barrio, y la misma distancia total que marcaba el codo.")
```
</details>

---


---
## Resumen en una frase 🎯

> ***K-Means* reparte los datos en un número de grupos que tú eliges, repitiendo dos pasos (cada punto va al centro más cercano y cada centro se muda al promedio de su grupo) hasta que nada cambie; siempre termina, pero conviene repetirlo varias veces, elegir bien el número de grupos y recordar que solo entiende grupos «redondos».**

### Lo que aprendiste hoy:
- ✅ Dos pasos que se repiten: **asignar** al centro más cercano y **mover** el centro al promedio.
- ✅ Siempre **termina**, pero puede quedar atascado en una solución **mala** según el inicio.
- ✅ Remedios: **repetir varias veces** y usar un **inicio inteligente** (*k-means++*).
- ✅ El **codo** ayuda a elegir cuántos grupos: ¿cuándo agregar uno más ya casi no mejora?
- ✅ Falla con **formas raras**, **tamaños distintos** y **clientes muy lejanos**.

> 🎓 **Siguiente paso:** [Clustering Jerárquico (Dummies)](02_Jerarquico_Dummies.ipynb). Versión con más detalle de este tema: [cuaderno estándar](../01_KMeans.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué *K-Means* siempre termina, pero no garantiza encontrar la mejor distribución de las tiendas? ¿Qué harías para reducir el riesgo?
2. Quieres agrupar fotos de un álbum en **dos tipos**: «fotos de día» y «fotos de noche», pero los datos forman dos **medias lunas** entrelazadas. ¿Por qué *K-Means* las separaría mal? ¿Qué tipo de método podría hacerlo mejor?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>